In [39]:
from dotenv import load_dotenv
load_dotenv()

True

In [40]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [41]:
loader=PyPDFLoader("../data/medical_report.pdf")
docs=loader.load()

In [42]:
len(docs)

9

In [43]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_docs=splitter.split_documents(docs)

In [44]:
len(splitted_docs)

26

In [45]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings,
    
)

In [46]:
same_record=vector_store.similarity_search("Patient name")

In [47]:
same_record[0].page_content

'Report Status    \nFemale\n27 Years:\n:\n:\n:\nAge\nGender\nReported        \nP\n9/7/2025   4:56:00PM\nDR NITIN NAHAR\n474764803\nMs. NIKITA  CHUDHARY:\n:\n:\n:\n:\nName        \nLab No.    \nRef By \nCollected       \nA/c Status \n10/7/2025  6:31:50PM\nFinal\nCollected at            : Processed at             :BHOPAL CC-82\nMr Rachel V John Pata So Vitus John Mig 26 \nGraund,Indrapuri, Phone: 8770817968\n \nLPL - Bhopal Lab II\nPlot No.05, Mandakini Housing Society, Near \nApurti Shopping Mall, Kolar Main Road, \nBhopal, M.P. -  462042\nTest Report      \nTest Name Results Units Bio. Ref. Interval\nHEMOGRAM\n(SLS Method, Sheath Flow DC Detection Method, Fluorescent Flow Cytometry, Calculated & Capillary Photometry)\nHemoglobin  12.00 - 15.00 g/dL13.10\nPacked Cell Volume (PCV)  36.00 - 46.00 %39.50\nRBC Count  3.80 - 4.80 mill/mm34.47\nMCV  83.00 - 101.00 fL88.40\nMentzer Index   19.8\nMCH  27.00 - 32.00 pg29.30\nMCHC  31.50 - 34.50 g/dL33.20\nRed Cell Distribution Width (RDW)  11.60

TO BULID AGENT = TOOLS, LLM, PROMPT

In [56]:
@tool
def retriever_tool(query:str):
    """
    This tool can help you to retrieve the relevant data of the PDF Documents, and these PDF 
    documents have details about medical reports.
    """

    print("Tool Called: ",query)
    docs=vector_store.similarity_search(query=query,k=4)
    context=""
    for doc in docs:
        context=doc.page_content + "\n\n"
    return context

In [57]:
retriever_tool.invoke("Patient_Name")

Tool Called:  Patient_Name


'Report Status    \nFemale\n27 Years:\n:\n:\n:\nAge\nGender\nReported        \nP\n09/07/2025  04:56:00PM\nDR NITIN NAHAR\n474764803\nMs. NIKITA  CHUDHARY:\n:\n:\n:\n:\nName        \nLab No.    \nRef By \nCollected       \nA/c Status Final\n10/7/2025  6:31:50PM\nCollected at            : Processed at             :BHOPAL CC-82\nMr Rachel V John Pata So Vitus John Mig 26 \nGraund,Indrapuri, Phone: 8770817968\n \nLPL - Bhopal Lab II\nPlot No.05, Mandakini Housing Society, \nNear Apurti Shopping Mall, Kolar Main \nRoad, Bhopal, M.P. -  462042\nTest Report \nTest Name Results Units Bio. Ref. Interval\n4.60Albumin  3.50 - 5.20 g/dL\n2.89Globulin(Calculated)  2.0 - 3.5 gm/dL\n1.59A : G Ratio  0.90 - 2.00 \n9.92Calcium, Total  8.60 - 10.30 mg/dL\n4.34Phosphorus  2.40 - 4.40 mg/dL\n137.31Sodium  136.00 - 146.00 mEq/L\n4.86Potassium  3.50 - 5.10 mEq/L\n103.66Chloride  101.00 - 109.00 mEq/L\nNote\n1. Estimated GFR  (eGFR) calculated using the 2021 CKD-EPI creatinine equation and GFR Category repor

In [58]:
llm=ChatGroq(model="llama-3.1-8b-instant")

In [59]:
system_prompt="""
    You are a helpful assistant that answers questions using retrieved context.
    ALWAYS use the 'retriever_tool' tool for questions requiring external knowledge
"""

In [60]:
agent=create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=system_prompt
)

In [61]:
query= "What is the name of patient and what is the name of doctors?"
respose=agent.invoke({"messages":[{"role":"user","content":query}]})

Tool Called:  Patient and doctor names in medical reports


In [62]:
reslt=respose["messages"][-1].content


In [63]:
print(reslt)

The patient's name is Ms. NIKITA  CHUDHARY, and the doctor's name is DR NITIN NAHAR.
